In [9]:
import sys 
import xarray as xr
import numpy as np
import os
import pandas as pd
import yaml
import math
from datetime import datetime, timedelta
from parcels import FieldSet, Field, VectorField, ParticleSet, JITParticle, ParcelsRandom, Variable,StatusCode,GeographicPolar, Geographic
from glob import glob
import matplotlib.pyplot as plt

import warnings

warnings.filterwarnings(
    "ignore",
    message="The specified chunks separate the stored chunks",
    category=UserWarning,
)

warnings.filterwarnings(
    "ignore",
    message="File .* could not be decoded properly by xarray",
    category=UserWarning,
)

In [10]:
sys.path.append(os.getcwd()+'/Source')

from OP_Kernels import *
from OP_functions import *

In [11]:
jjii = xr.open_dataset('~/MOAD/grid/grid_from_lat_lon_mask999.nc')

def finder(jjii,lati,loni):
    j = [jjii.jj.sel(lats=lati, lons=loni, method='nearest').item()][0]
    i = [jjii.ii.sel(lats=lati, lons=loni, method='nearest').item()][0]
    return j,i

finder(jjii,49.35844243668596,-123.80204117521916)

(512, 251)

In [12]:
coord = xr.open_dataset(path()['coords'],decode_times=False)
mask = xr.open_dataset(path()['mask'],decode_times=False)

In [13]:
#clat,clon = p_unidist(coord.nav_lat,coord.nav_lon,mask.totaldepth,20,20)
# np.save('lat.npy',np.array(clat))
# np.save('lon.npy',np.array(clon))

In [14]:
clat = np.load('lat.npy')
clon = np.load('lon.npy')

In [15]:
cz = np.ones(len(clat))

In [8]:
param = load_config('yaml/config.yaml')
start = datetime(param['startdate']['year'], param['startdate']['month'], param['startdate']['day']) #Start date
length = param['param']['length'] # Set Time length [days] 
dt = param['param']['dt'] #toggle between - or + to pick backwards or forwards 
N = param['particle']['filename'] # Name of deploy loc file
dd = param['particle']['dd'] #max depth difference z in N
name = param['file']['name'] #name output file
dtp = param['param']['dtp'] #how often particle released in hours
odt = param['param']['odt'] #how often data is recorded
rrr = param['particle']['r'] #radious of particle deployment
SI = param['constants']['SI'] #how many particles per super-individual

length = 1
duration = timedelta(days=length)
print(f"The model will run for {duration.days} days, starting at {start}")

n = 1 
N = len(clat)
deg2m = 111319.5 * np.cos(49 * np.pi / 180)
var = (rrr / (deg2m * 3))**2
x_offset, y_offset = np.random.multivariate_normal([0, 0], [[var, 0], [0, var]], [n,N]).T #Here we define a cloud of deployment points.
z_offset=np.random.random_sample([N]).T*(dd) #add some depth variability
lon = np.zeros([N])
lat = np.zeros([N])
z = np.zeros([N])
for i in range(N):
    lon[i]=(clon[i] + x_offset[i])
    lat[i]=(clat[i] + y_offset[i])
    z[i]=(cz[i] + z_offset[i])


print(f"The model will release {N} particle every timestep")

daterange = [start+timedelta(days=i) for i in range(length)]
fn =  name + '.zarr'
outfile = os.path.join(os.getcwd()+'/results/', fn)

print(f"Output file: {outfile}")

The model will run for 1 days, starting at 2024-01-01 00:00:00
The model will release 217 particle every timestep
Output file: /home/jvalenti/MOAD/analysis-jose/notebooks/MP_model/results/test_function.zarr


/tmp/ipykernel_626398/951340373.py:27: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  lon[i]=(clon[i] + x_offset[i])
/tmp/ipykernel_626398/951340373.py:28: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  lat[i]=(clat[i] + y_offset[i])


In [9]:
cs = {
        "time": ("time", 1),
        "depth": ("depth", 40),   
        "lat": ("latitude", 100),
        "lon": ("longitude", 100),
    }

In [10]:
duration = timedelta(days=length)
paths = path()
cgrid_dim = {'lon': 'glamf','lat': 'gphif','depth':'depthw', 'time':'time_counter'}
Ulist, Vlist, Wlist, Tlist, Blist, Plist, Glist ,Waveslist, Windlist = [], [], [], [], [], [], [], [], []

for day in range(duration.days):
  path_NEMO = make_prefix(start + timedelta(days=day),paths['NEMO'])
  Ulist.append(path_NEMO + '_grid_U.nc')
  Vlist.append(path_NEMO + '_grid_V.nc')
  Wlist.append(path_NEMO + '_grid_W.nc')
  Tlist.append(path_NEMO + '_grid_T.nc')

  Glist.append(get_Biol_path(start + timedelta(days=day),'graz'))
  Plist.append(get_Biol_path(start + timedelta(days=day),'prod'))
  Blist.append(get_Biol_path(start + timedelta(days=day)))
  
  Waveslist.append(get_WW3_path(start + timedelta(days=day)))
  Windlist.append(get_Winds_path(start + timedelta(days=day)))


filenames = {
  'U': {'lon': paths['coords'],'lat': paths['coords'],'depth': Wlist[0],'data': Ulist},
  'V': {'lon': paths['coords'],'lat': paths['coords'],'depth': Wlist[0],'data': Vlist},
  'W': {'lon': paths['coords'],'lat': paths['coords'],'depth': Wlist[0],'data': Wlist}
}
dimensions = {
  'U': {'lon': 'glamf','lat': 'gphif','depth':'depthw', 'time':'time_counter'},
  'V': {'lon': 'glamf','lat': 'gphif','depth':'depthw', 'time':'time_counter'},
  'W': {'lon': 'glamf','lat': 'gphif','depth':'depthw', 'time':'time_counter'},
}
variables = {
  'U': 'vozocrtx',
  'V': 'vomecrty',
  'W': 'vovecrtz'
}

field_set = FieldSet.from_nemo(
    filenames,
    variables,
    dimensions,
    allow_time_extrapolation=True,
    chunksize=cs
)

dimensions['Kz'] = {'lon': 'glamt', 'lat': 'gphit', 'depth': 'depthw','time': 'time_counter'}
Kz = Field.from_netcdf(filenames['W'], 'vert_eddy_diff', dimensions['Kz'],allow_time_extrapolation=True, chunksize=cs)
field_set.add_field(Kz)

filenames['tracer'] = {'lon': paths['coords'],'lat': paths['coords'],'depth': Tlist[0],'data': Tlist}
dimensions['tracer'] = {'lon': 'glamt', 'lat': 'gphit', 'depth': 'deptht','time': 'time_counter'}

R = Field.from_netcdf(filenames['tracer'], 'sigma_theta', dimensions['tracer'],allow_time_extrapolation=True, chunksize=cs)
T = Field.from_netcdf(filenames['tracer'], 'votemper', dimensions['tracer'],allow_time_extrapolation=True, chunksize=cs)

field_set.add_field(R)
field_set.add_field(T)

#BIOLOGY fields
filenames['Biology'] = {'lon': paths['coords'],'lat': paths['coords'],'depth': Blist[0],'data': Blist}
filenames['Prod'] = {'lon': paths['coords'],'lat': paths['coords'],'depth': Plist[0],'data': Plist}
filenames['Mort'] = {'lon': paths['coords'],'lat': paths['coords'],'depth': Glist[0],'data': Glist}

PP = Field.from_netcdf(filenames['Prod'], 'PPDIAT', dimensions['tracer'],allow_time_extrapolation=True, chunksize=cs)
HBProd = Field.from_netcdf(filenames['Prod'], 'HetHBAC', dimensions['tracer'],allow_time_extrapolation=True, chunksize=cs)
Diat_mort = Field.from_netcdf(filenames['Mort'], 'MORTDIAT', dimensions['tracer'],allow_time_extrapolation=True, chunksize=cs)
Hbac_mort = Field.from_netcdf(filenames['Mort'], 'MORTHBAC', dimensions['tracer'],allow_time_extrapolation=True, chunksize=cs)
Hbac = Field.from_netcdf(filenames['Biology'], 'heterotrophic_bacteria', dimensions['tracer'],allow_time_extrapolation=True, chunksize=cs)
Diat = Field.from_netcdf(filenames['Biology'], 'diatoms', dimensions['tracer'],allow_time_extrapolation=True, chunksize=cs)

field_set.add_field(PP)
field_set.add_field(HBProd)
field_set.add_field(Diat_mort)
field_set.add_field(Hbac_mort)
field_set.add_field(Hbac)
field_set.add_field(Diat)

#####FROM HERE ALL NOT CHUNKED FIELDS (2D or different dimensions)
#Add Stokes Drift fields
dimensions['stokes'] = {'lon': 'longitude', 'lat': 'latitude', 'time': 'time'}
filenames['stokes'] = {'lon': paths['coordsWW3'], 'lat': paths['coordsWW3'], 'data': Waveslist}
us = Field.from_netcdf(filenames['stokes'],'uuss', dimensions['stokes'],allow_time_extrapolation=True)
vs = Field.from_netcdf(filenames['stokes'], 'vuss', dimensions['stokes'],allow_time_extrapolation=True)
wl = Field.from_netcdf(filenames['stokes'], 'lm', dimensions['stokes'],allow_time_extrapolation=True)
us.units = GeographicPolar()   # u: m/s -> deg lon/s 
vs.units = Geographic() 
field_set.add_field(us)
field_set.add_field(vs)
field_set.add_field(wl)
field_set.add_vector_field(VectorField("stokes", us, vs, wl))

filenames['totaldepth'] = {'lon': paths['coords'], 'lat': paths['coords'], 'data': paths['mask']}
filenames['ssh'] = {'lon': paths['coords'], 'lat': paths['coords'], 'data': Tlist}
variables['totaldepth']= 'totaldepth'
variables['ssh']= 'sossheig'

#Add Bathymetry 2D field
dimensions['flat'] = {'lon': 'glamt', 'lat': 'gphit'}
TD = Field.from_netcdf(filenames['totaldepth'], variables['totaldepth'], dimensions['flat'],allow_time_extrapolation=True)
field_set.add_field(TD)

#Add SSH 
dimensions['flat_t'] = {'lon': 'glamt', 'lat': 'gphit','time': 'time_counter'}
SSH = Field.from_netcdf(filenames['ssh'], variables['ssh'], dimensions['flat_t'],allow_time_extrapolation=True)
field_set.add_field(SSH)

b_stress = Field.from_netcdf(filenames['Biology'], 'bottom_stress', dimensions['flat_t'],allow_time_extrapolation=True)
field_set.add_field(b_stress)

filenames['wind'] = {'lon': Windlist[0],'lat': Windlist[0],'data': Windlist}
dimensions['wind'] = {'lon': 'nav_lon', 'lat': 'nav_lat','time': 'time_counter'}

u_wind = Field.from_netcdf(filenames['wind'], 'u_wind', dimensions['wind'],allow_time_extrapolation=True)
v_wind = Field.from_netcdf(filenames['wind'], 'v_wind', dimensions['wind'],allow_time_extrapolation=True)
u_wind.units = GeographicPolar()   # u: m/s -> deg lon/s 
v_wind.units = Geographic()  
field_set.add_field(u_wind)
field_set.add_field(v_wind)
field_set.add_vector_field(VectorField("wind", u_wind, v_wind))

## Define constants

In [11]:
#Define our Parcels Particles
class MPParticle(JITParticle):    
    fact = Variable('fact', initial = 1.0)
    status = Variable('status', initial = 1) #0=non initialized, 1=active, 2=beached, 3 = sedimented, 5=export to PO, 10=lost
    N_hb = Variable('N_hb', initial = 0)  #n of hbac attached
    N_ph = Variable('N_ph', initial = 0)  #n of diat attached
    bioth = Variable('bioth', initial = 0.0, to_write=False)  #thickness biofouling (can be calculated from N and mp dimensions)
    #these do not change change in run
    dim1 = Variable('dim1', initial = 7.5e-6, to_write=False) # particle small semiaxis
    dim2= Variable('dim2', initial = 7.5e-6, to_write=False)  # particle medium semiaxis
    dim3 = Variable('dim3', initial = 2e-4, to_write=False) # particle large semiaxis
    density = Variable('density', initial = 1350, to_write=False) #particle density 
    shape = Variable('shape', initial = 1.0, to_write=False) # particle roundness (0 is cube, 1 is sphere) 
    Vol = Variable('Vol', initial = 1e-13, to_write=False)  #estimated at the initialization
    ws = Variable('ws', initial=0.0)
    last_update_biol = Variable('last_update_biol', initial=-5000.0, to_write=False) #careful with initialization
    last_update_dyn = Variable('last_update_dyn', initial=-100.0, to_write=False)
    vtau_crit = Variable('vtau_crit', initial=1.0, to_write=False)

In [12]:
field_set.add_constant('deg2met', param['constants']['deg2met'])
field_set.add_constant('deglatT', param['constants']['deglatT'])   
field_set.add_constant('kh', param['constants']['kh'])  
field_set.add_constant('rho_hb', param['constants']['rho_hb']) 
field_set.add_constant('rho_ph', param['constants']['rho_ph']) 
field_set.add_constant('V_hb', param['constants']['V_hb'])
field_set.add_constant('V_ph', param['constants']['V_ph']) 
field_set.add_constant('gravity', param['constants']['gravity']) 
field_set.add_constant('biol_dt', param['constants']['biol_dt']) 
field_set.add_constant('dyn_dt', param['constants']['dyn_dt']) 
field_set.add_constant('kB', param['constants']['kB']) 
field_set.add_constant('N_hb_cell', param['constants']['N_hb_cell'])
field_set.add_constant('N_ph_cell', param['constants']['N_ph_cell']) 
field_set.add_constant('r_hbac', param['constants']['r_hbac']) 
field_set.add_constant('r_phyt', param['constants']['r_phyt']) 
field_set.add_constant('vs_hbac', param['constants']['vs_hbac']) 
field_set.add_constant('vs_diat', param['constants']['vs_diat']) 
field_set.add_constant('resusp_h', param['constants']['resusp_h']) 
PrUb = 1- np.exp(-param['param']['dt']/(param['constants']['Ub']*86400))
field_set.add_constant('PrUb', PrUb)

## Define Sources

In [13]:
lat = np.ones(len(clat))*49.33451996322204
lon = np.ones(len(clat))*-123.83734250024288
z = np.ones(len(clat))*0.01

#Define dimensions before the run (skip initialization and save output space) a,b,c,e,vol,density
#Vol = 2.0*a*b*c * (e**2) * math.gamma(e/2 + 1)*math.gamma(e)/math.gamma(3*e/2 + 1) * math.gamma(e/2)*math.gamma(e/2)/math.gamma(e)
#SD = Variable('SD', initial = 3.34e-6, to_write=False) #std Diameter only used in init
#SDL = Variable('SDL', initial = 24.4e-5, to_write=False) #std length only used in init

In [14]:
pset = ParticleSet.from_list(field_set, MPParticle, lon=lon, lat=lat, depth=z)

In [15]:
pset.execute(
    [vert_mix,Advection,horz_mix,Stokes_Windage,Biofouling_ws,Buoyancy,export,KeepInOcean,CheckOutOfBounds],
    runtime=timedelta(hours=12),
    dt=timedelta(seconds=dt),
    output_file=pset.ParticleFile(name='/home/jvalenti/MOAD/results/output.zarr', outputdt=timedelta(minutes=30)))

INFO: Output files are stored in /home/jvalenti/MOAD/results/output.zarr.
100%|██████████| 43200.0/43200.0 [01:26<00:00, 501.49it/s]
